In [3]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping


ModuleNotFoundError: No module named 'sklearn'

In [ ]:
# Load data
_df = pd.read_csv('test_results_cleaned.csv')

# Drop non-numeric identifier columns
_df = _df.drop(columns=['Test ID'], errors='ignore')

# Split input/target
X = _df.drop(columns=['Suspicious'])
y = _df['Suspicious']

# Stratified cross-validation setup
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Prepare a simple evaluation loop
fold_scores = []
for train_index, test_index in skf.split(X, y):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    X_train = X_train.reshape((X_train.shape[0], 1, X_train.shape[1]))
    X_test = X_test.reshape((X_test.shape[0], 1, X_test.shape[1]))

    model = Sequential([
        LSTM(64, input_shape=(X_train.shape[1], X_train.shape[2])),
        Dropout(0.3),
        Dense(32, activation='relu'),
        Dropout(0.2),
        Dense(1, activation='sigmoid')
    ])

    model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
    es = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

    history = model.fit(
        X_train,
        y_train,
        validation_split=0.2,
        epochs=50,
        batch_size=32,
        callbacks=[es],
        verbose=0
    )

    loss, acc = model.evaluate(X_test, y_test, verbose=0)
    fold_scores.append(acc)
    print(f'Fold {len(fold_scores)} accuracy: {acc:.4f}')

print(f'Average accuracy across folds: {np.mean(fold_scores):.4f}')
